# Transition fitting + finite-size extrapolation, one field cut at a time

**Goal:** locate the topological→trivial transition of the 3D bosonic toric code along a
*cut* — two of $(h_x, h_y, h_z)$ fixed, the third swept — at every available $L$, then
extrapolate $h_c(L)\to h_c(\infty)$. The result is banked as
`results/transitions/<tag>.json` (`WRITE_JSON = True` in the config cell; off by default so
Run-All never overwrites the banked records). Vetted values are copied by hand into
`phase_diagram_manual.ipynb`; `phase3d_L4_planes.ipynb` reads the `hy_cuts_L4` records.

All fitting/FSS code lives in **`analysis/scripts/transition_fit.py`** (NetKet-free);
this notebook is the driver. Two data lanes:

| lane | source | order parameter | $L$ |
|---|---|---|---|
| `prod` (2026-08, dual-basis 500-step protocol) | `results/phaseB*/` ($h_y{=}0$), `results/hy_cuts_L4/` ($h_y{=}0.2,0.4$) — per-run final JSONs, **lowest-energy non-diverged run wins per point** | Z-string `O_FM_paratoric` (electric cut) / X-membrane `O_FM_membrane_R1` (magnetic cut) | 4,5,6 at $h_y{=}0$; **4 only** at $h_y\neq0$ |
| `old` (2026-07 campaign, pre-optimization stack) | `data/tc_nqs/phase_h*/fm_L*.json` (gitignored mirror; skipped if absent) | `bulkR1` loop / `memR*` membrane | 4–7 |

Usage: pick `CUT` in §1 and run top to bottom; set `RUN_ALL = True` in §7 to (re)bank
every cut in the registry. Physics conventions: $O_{\rm FM}=0$ topological → 1 trivial;
exact thermodynamic anchors at $h_y=0$: $h_z^c(h_x{=}0)=0.193869$ (2nd order, 3D-Ising$^*$),
$h_x^c(h_z{=}0)=1$ (1st order).

In [ ]:
import json, os, sys
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path

ROOT = (Path.cwd().parents[1] if Path.cwd().name == "notebooks" else Path.cwd()).resolve()
sys.path.insert(0, str(ROOT / "analysis" / "scripts"))
import transition_fit as tf

RES, DATA, OUT = ROOT / "results", ROOT / "data" / "tc_nqs", ROOT / "results" / "transitions"
FIGS = ROOT / "analysis" / "figs"
NU = tf.NU_3D_ISING                     # 0.62997 -> 1/nu = 1.587

# ---- knobs ----
SAVE_FIGS  = False                      # house rule: savefig lines stay commented; promote by hand
WRITE_JSON = False                      # True = (re)bank results/transitions/<tag>.json; keep False for Run-All
ERR_X      = 3                          # bars x3 on order-parameter panels (house rule)
ERR_MODE   = "pdg"                      # per-L fit errors: 'pdg' (inflate by sqrt(chi2red) if >1) | 'scaled' | 'absolute'

plt.rcParams.update({"figure.dpi": 110, "font.size": 10})
print("ROOT =", ROOT, "| old lane present:", DATA.exists())

## 1 · Cut registry

One entry per cut. `runs` (prod lane) or `fm_dir` + `fm_tags` (old lane); `window` = field
range handed to the fits; optional `kind="trivial-trivial"` / `secondary=[...]` pass through
to the record (campaign contract, 2026-09-09); `x_main` / `xs` = the FSS exponent and the exponent battery whose
$h_c(\infty)$ spread is quoted as the exponent systematic. Defaults: 2nd-order cuts use
$x=1/\nu_{\rm 3D\,Ising}$ as primary with $\{1, 1/\nu, 2\}$; 1st-order cuts use $x=1$
(OBC surface shift) with $\{1,2,3\}$.

In [ ]:
def _cut(hy, fixed, sweep, order, **kw):
    x2, x1 = (1 / NU, (1.0, 1 / NU, 2.0)), (1.0, (1.0, 2.0, 3.0))
    spec = dict(hy=hy, fixed=fixed, sweep=sweep, order=order, lane=kw.pop("lane", "prod"))
    spec["x_main"], spec["xs"] = x2 if order == 2 else x1
    spec.update(kw)
    return tf.cut_tag(hy, *fixed, sweep, spec["lane"]), spec

CUTS = tf.register_cuts([
    # ---- h_y = 0, production lane (Phase B + reconciliation rerun; L = 4, 5, 6) ----
    _cut(0.0, ("hx", 0.2), "hz", 2, obs="O_FM_paratoric", window=(0.1, 0.5),
         runs=[RES / f"phaseB/up/L{L}" for L in (4, 5, 6)] + [RES / f"phaseB_rerun/up/L{L}" for L in (4, 5, 6)]),
    _cut(0.0, ("hz", 0.1), "hx", 1, obs="O_FM_membrane_R1", window=(0.5, 1.3),
         runs=[RES / f"phaseB/right/L{L}" for L in (4, 5, 6)] + [RES / f"phaseB_rerun/right/L{L}" for L in (4, 5, 6)]),
    # ---- h_y = 0.2 / 0.4, sign-full lane (hy-cuts campaign 2026-08-26; L = 4 only) ----
    *[_cut(hy, ("hx", 0.2), "hz", 2, obs="O_FM_paratoric", window=(0.1, 0.5), s2_from="snapshots",
           runs=[RES / f"hy_cuts_L4/up/hy{hy}/L4"]) for hy in (0.2, 0.4)],
    *[_cut(hy, ("hz", 0.1), "hx", 1, obs="O_FM_membrane_R1", window=(0.5, 1.3),
           runs=[RES / f"hy_cuts_L4/right/hy{hy}/L4"]) for hy in (0.2, 0.4)],
])
if DATA.exists():   # ---- h_y = 0, pre-optimization lane (2026-07; L = 4..7); biased high, kept for coverage ----
    CUTS = tf.register_cuts(CUTS,
        [_cut(0.0, ("hx", hx), "hz", 2, lane="old", window=(0.1, 0.4),
              fm_dir=DATA / f"phase_hx{hx}", fm_tags=("bulkR1",)) for hx in (0.0, 0.2, 0.4, 0.6, 0.8, 1.0)]
        + [_cut(0.0, ("hz", hz), "hx", 1, lane="old", window=(0.4, 1.3),
                fm_dir=DATA / f"phase_hz{hz}", fm_tags=("memR1", "memR2", "memA0.5"))
           for hz in (0.0, 0.1, 0.2, 0.3, 0.4, 0.5, 0.7, 0.9, 1.0, 1.1)])

# ---- campaign cuts (results/phase3d/hy{hy}/{electric_hx*|magnetic_hz*}/L*; appended by the
# ---- phase-diagram campaign, ONE _cut(...) per line inside CAMPAIGN; kind= / secondary= pass through) ----
CAMPAIGN = [
    # phase3d campaign, hy=0 electric line (fixed hx, sweep hz; L=4/5/6; obs=O_FM_paratoric)
    *[_cut(0.0, ("hx", float(hxs)), "hz", 2, obs="O_FM_paratoric", window=(0.1, 0.55), lane="phase3d",
           runs=[RES / f"phase3d/hy0.0/electric_hx{hxs}/L{L}" for L in (4, 5, 6)])
      for hxs in ("0.0", "0.5", "0.8")],
]
CUTS = tf.register_cuts(CUTS, CAMPAIGN)     # raises on a duplicate tag (collision guard)

CUT = "hy0_hx0_sweep-hz@phase3d"        # <- pick the cut to analyse below (phase3d electric hx=0.0)
print(f"{len(CUTS)} cuts registered; CUT = {CUT}")
for t, s in CUTS.items():
    print(f"  {t:32s} order={s['order']} lane={s['lane']}")

## 2 · Load the cut → per-$L$ curves

`load_cut` returns `{"O": {L: Curve}, "S2": {L: Curve}, "E": {L: Curve}}`. Prod lane:
S2 comes from the final-state `observables.S2` when present, else from the `*.snapshots.json`
replay series (h_y ≠ 0 runs). Energies are the winners' `E0` (per spin: $N = 3L^3-3L^2$ OBC).

In [ ]:
def n_spins(L):                                   # OBC edge count
    return 3 * L**3 - 3 * L**2

def load_cut(spec):
    fixed = {spec["fixed"][0]: spec["fixed"][1], "hy": spec["hy"]}
    if spec["lane"] == "old":
        O = tf.load_fm_jsons(spec["fm_dir"], spec["fm_tags"])
        S2 = tf.load_s2_jsons(spec["fm_dir"])
        E = tf.load_energy_jsons(spec["fm_dir"])
        return {"O": O, "S2": S2, "E": E}
    O = tf.load_runs(spec["runs"], spec["sweep"], fixed, spec["obs"])
    S2 = (tf.load_snapshot_s2(spec["runs"], spec["sweep"], fixed) if spec.get("s2_from") == "snapshots"
          else tf.load_runs(spec["runs"], spec["sweep"], fixed, "S2"))
    E = {L: tf.Curve(L, c.h, c.E, np.full_like(c.h, np.nan), "E") for L, c in O.items()}
    return {"O": O, "S2": S2, "E": E}

spec = CUTS[CUT]
data = load_cut(spec)
print(f"{CUT}: sweep {spec['sweep']} | fixed {dict([spec['fixed']])} hy={spec['hy']} | order {spec['order']}")
for L, c in data["O"].items():
    s2n = len(data["S2"].get(L, tf.Curve(L, [], [], [], 'S2')).h)
    print(f"  L={L}: {len(c.h):2d} O_FM points  [{c.h.min():.3g}, {c.h.max():.3g}]  | {s2n:2d} S2 points | {c.src.split(';')[0]}")

In [ ]:
# %% curves per L: order parameter, S2, energy per spin
COL = tf.plasma_by_L(list(data["O"]))
sw = spec["sweep"]
fig, axes = plt.subplots(1, 3, figsize=(12, 3.5))
for L, c in data["O"].items():
    axes[0].errorbar(c.h, c.y, ERR_X * np.nan_to_num(c.ye), fmt="o", color=COL[L], ms=5, capsize=2, lw=1, label=f"L={L}")
for L, c in data["S2"].items():
    axes[1].errorbar(c.h, c.y, ERR_X * np.nan_to_num(c.ye), fmt="o", color=COL[L], ms=5, capsize=2, lw=1, label=f"L={L}")
axes[1].axhline(3 * np.log(2), color="0.75", ls="--", lw=1, label=r"$3\ln2$ (topo, exact)")
for L, c in data["E"].items():
    axes[2].plot(c.h, c.y / n_spins(L), "o-", color=COL[L], ms=4, lw=1, label=f"L={L}")
axes[0].set_ylabel(rf"$O_\mathrm{{FM}}$  (bars $\times${ERR_X})")
axes[1].set_ylabel(rf"$S_2$ central plaquette  (bars $\times${ERR_X})")
axes[2].set_ylabel("$E/N$")
for ax in axes:
    ax.set_xlabel(f"$h_{sw[1]}$"); tf.openax(ax); ax.legend(frameon=False, fontsize=8)
fig.suptitle(f"{CUT}: raw curves", y=1.02); fig.tight_layout()
# if SAVE_FIGS: plt.savefig(FIGS / f"fss_{CUT}_curves.png", dpi=300, bbox_inches="tight")
plt.show()

## 3 · Per-$L$ transition locators

Four locators per curve, all in `transition_fit`:

* **logistic** — 4-parameter sigmoid, $h_c$ = inflection $h_0$ (the `tc3d.fm` convention);
* **richards** — 6-parameter generalized logistic (asymmetric rise), inflection
  $M+\ln(Q/\nu)/B$ and mid-rise $M+\ln(Q/(2^\nu-1))/B$;
* **fd_peak** — model-free: parabola through the three finite-difference slopes around
  $\max|dO/dh|$, bootstrap error floored at ¼ grid step;
* **S2** — logistic inflection of the central-plaquette Rényi entropy (independent locator).

The raw MC bars under-cover the curve's systematics ($\chi^2_{\rm red}\sim 50$–$150$ on the
prod lane — same reason the panels show bars ×3), so `ERR_MODE="pdg"` inflates every fit
error by $\max(1,\sqrt{\chi^2_{\rm red}})$.

In [ ]:
FITS, S2FITS = {}, {}
for L, c in data["O"].items():
    FITS[L] = tf.locate_all(c, spec["window"], ERR_MODE)
    S2FITS[L] = tf.fit_logistic(data["S2"][L], ERR_MODE, spec["window"]) if L in data["S2"] else None

print(f"{'L':>2} {'logistic':>16} {'chi2r':>6} {'richards infl':>16} {'rich. half':>10} {'fd peak':>16} {'S2 infl':>16}")
for L, f in FITS.items():
    lg, rc, fd, s2 = f["logistic"], f["richards"], f["fd_peak"], S2FITS[L]
    fmt = lambda x: f"{x.h_c:.4f}({x.h_c_err:.4f})" if (x is not None and x.ok()) else "     —    "
    half = f"{rc.extra['h_half']:.4f}" if rc.ok() else "   —  "
    print(f"{L:>2} {fmt(lg):>16} {lg.chi2red:6.1f} {fmt(rc):>16} {half:>10} {fmt(fd):>16} {fmt(s2):>16}")

In [ ]:
# %% fits overlay: logistic (solid) + Richards (dotted); vertical = logistic inflection
fig, axes = plt.subplots(1, 2, figsize=(10, 3.8))
lo, hi = spec["window"]
xx = np.linspace(lo, hi, 400)
for L, c in data["O"].items():
    lg, rc = FITS[L]["logistic"], FITS[L]["richards"]
    axes[0].errorbar(c.h, c.y, ERR_X * np.nan_to_num(c.ye), fmt="o", color=COL[L], ms=4, capsize=2, lw=0, elinewidth=1, label=f"L={L}")
    if lg.ok():
        axes[0].plot(xx, tf.logistic(xx, *lg.popt), "-", color=COL[L], lw=1.2)
        axes[0].axvline(lg.h_c, color=COL[L], ls=":", lw=1)
        axes[1].plot(xx, tf.dlogistic(xx, *lg.popt), "-", color=COL[L], lw=1.2, label=f"L={L} logistic")
    if rc.ok():
        axes[0].plot(xx, tf.richards(xx, *rc.popt), ":", color=COL[L], lw=1.2)
    hm = 0.5 * (c.h[1:] + c.h[:-1])
    axes[1].plot(hm, np.diff(c.y) / np.diff(c.h), "o", color=COL[L], ms=3, alpha=0.6)
for L, s2 in S2FITS.items():
    if s2 is not None and s2.ok():
        axes[1].axvline(s2.h_c, color=COL[L], ls="--", lw=0.8, alpha=0.7)
axes[0].set(xlabel=f"$h_{sw[1]}$", ylabel=rf"$O_\mathrm{{FM}}$  (bars $\times${ERR_X})", xlim=(lo, hi))
axes[1].set(xlabel=f"$h_{sw[1]}$", ylabel=r"$dO_\mathrm{FM}/dh$  (dots: finite diff.; dashed: $S_2$ inflection)", xlim=(lo, hi))
for ax in axes: tf.openax(ax); ax.legend(frameon=False, fontsize=8)
fig.suptitle(f"{CUT}: per-L locators", y=1.02); fig.tight_layout()
# if SAVE_FIGS: plt.savefig(FIGS / f"fss_{CUT}_fits.png", dpi=300, bbox_inches="tight")
plt.show()

## 4 · Combining the locators into one $h_c(L)$ — *policy cell*

Which number is "the" transition at a given $L$ is a modelling choice, not a fit output:
the logistic and Richards inflections differ by ~0.02 on the prod lane (asymmetric rise),
the finite-difference peak sits lower still, and the S2 locator has no phase-coherence
error diagnostic at $h_y\neq0$ (bars optimistic). **Policy in force (2026-09-09):
central = Richards inflection** — the OBC curves rise asymmetrically (slow onset, fast
saturation), which the symmetric logistic absorbs by shifting its inflection up by ~0.02;
Richards fits that shape and agrees with the S2 locator at $L=4$. Falls back to the
logistic if Richards does not converge. Error = PDG-inflated statistical ⊕ half-spread of
all other converged locators (systematic). Override `combine_markers` here to change the
policy — e.g. average the locators or require S2 agreement.

In [ ]:
def combine_markers(fits, s2_fit=None):
    """Per-L marker policy -> (h_c, err, info): Richards inflection central (see §4)."""
    return tf.combine_default(fits, s2_fit, central="richards")

ROWS = []
for L in FITS:
    h_c, err, info = combine_markers(FITS[L], S2FITS[L])
    ROWS.append({"L": L, "h_c": h_c, "h_c_err": err, **info,
                 "locators": {m: [f.h_c, f.h_c_err, f.chi2red] for m, f in FITS[L].items()},
                 "S2": None if S2FITS[L] is None or not S2FITS[L].ok() else [S2FITS[L].h_c, S2FITS[L].h_c_err, S2FITS[L].chi2red],
                 "n_points": int(len(data["O"][L].h)), "amp": float(FITS[L]["logistic"].popt[1]) if FITS[L]["logistic"].ok() else None})
print(f"{'L':>2} {'h_c(L)':>8} {'± total':>8} {'stat':>7} {'syst':>7}  spread over")
for r in ROWS:
    print(f"{r['L']:>2} {r['h_c']:8.4f} {r['h_c_err']:8.4f} {r.get('stat', np.nan):7.4f} {r.get('syst', np.nan):7.4f}  {np.round(r.get('spread_over', []), 4)}")

## 5 · Finite-size extrapolation

$h_c(L) = h_c(\infty) + a\,L^{-x}$, weighted by the §4 errors. Primary exponent `x_main`
(2nd order: $1/\nu_{\rm 3D\,Ising}=1.587$; 1st order: $x=1$, OBC surface term); the battery
`xs` gives the exponent systematic (half-spread of $h_c(\infty)$). A free-$x$ fit is reported
when ≥3 sizes exist (exactly determined at 3 — indicative only). Pairwise crossings of the
fitted logistics are printed as a size-independence check (for a scaling order parameter
they would coincide at $h_c$; $O_{\rm FM}$ is not one, so read them as a trend only).

In [ ]:
Ls  = [r["L"] for r in ROWS if np.isfinite(r["h_c"])]
hc  = [r["h_c"] for r in ROWS if np.isfinite(r["h_c"])]
hce = [r["h_c_err"] for r in ROWS if np.isfinite(r["h_c"])]

FSS  = tf.fss_fit(Ls, hc, hce, spec["x_main"], ERR_MODE)
SWP  = tf.fss_sweep(Ls, hc, hce, spec["xs"])
FREE = tf.fss_free(Ls, hc, hce) if len(Ls) >= 3 else None
syst_x = 0.5 * (max(s["h_inf"] for s in SWP) - min(s["h_inf"] for s in SWP)) if len(Ls) >= 2 else np.nan

if len(Ls) >= 2:
    print(f"h_c(inf) [x={FSS['x']:.3f}] = {FSS['h_inf']:.4f} ± {FSS['h_inf_err']:.4f} (stat) ± {syst_x:.4f} (exponent)   chi2red={FSS['chi2red']:.2f}  n={FSS['n']}")
    for s in SWP:
        print(f"   x={s['x']:.3f}: h_inf={s['h_inf']:.4f}({s['h_inf_err']:.4f})  a={s['a']:+.3f}  chi2red={s['chi2red']:.2f}")
    if FREE: print(f"   free x: x={FREE['x']:.2f}±{FREE.get('x_err', np.nan):.2f}  h_inf={FREE['h_inf']:.4f}({FREE['h_inf_err']:.4f})  {FREE.get('note','')}")
    print("   pairwise logistic crossings:", {k: round(v, 4) for k, v in tf.pairwise_crossings({L: FITS[L]['logistic'] for L in FITS}, spec['window']).items()})
else:
    print(f"only {len(Ls)} size(s): no extrapolation; h_c(L={Ls[0]}) = {hc[0]:.4f} ± {hce[0]:.4f} is the banked pseudo-critical point")

fig, ax = plt.subplots(figsize=(5.2, 3.8))
Ls_a = np.array(Ls, float)
if len(Ls) >= 2:
    XS = np.linspace(0, max(Ls_a ** -FSS["x"]) * 1.1, 100)
    ax.plot(XS, FSS["h_inf"] + FSS["a"] * XS, "-", color="0.3", lw=1.2, label=rf"$x={FSS['x']:.3f}$: $h_c(\infty)={FSS['h_inf']:.3f}({FSS['h_inf_err']:.3f})$")
    ax.axhspan(FSS["h_inf"] - syst_x, FSS["h_inf"] + syst_x, color="0.85", label=f"exponent band x∈{tuple(round(x, 2) for x in spec['xs'])}")
    ax.errorbar(0, FSS["h_inf"], FSS["h_inf_err"], fmt="s", color="0.3", ms=6, capsize=3)
for L, h, e in zip(Ls, hc, hce):
    ax.errorbar(L ** -FSS["x"] if len(Ls) >= 2 else 1 / L, h, e, fmt="o", color=COL[L], mec="k", mew=0.4, ms=6, capsize=2, label=f"L={L}")
ax.set(xlabel=rf"$L^{{-x}}$", ylabel=rf"$h_{sw[1]}^c(L)$")
tf.openax(ax); ax.legend(frameon=False, fontsize=8)
fig.suptitle(f"{CUT}: finite-size extrapolation", y=1.02); fig.tight_layout()
# if SAVE_FIGS: plt.savefig(FIGS / f"fss_{CUT}_extrap.png", dpi=300, bbox_inches="tight")
plt.show()

## 6 · Bank the record

`results/transitions/<tag>.json`: per-$L$ markers with all locators, the primary FSS, the
exponent battery, the free-$x$ fit, and a `quality` block (every size converged, $h_c$ inside
the window interior, rise amplitude > 0.2) that flags unreliable cuts.

In [ ]:
def bank(tag, spec, rows, fss, swp, free, syst_x):
    """tf.make_record adds syst_exponent, the quality gate, kind and the window/exponent spec."""
    return tf.make_record(tag, spec, rows, fss, swp, free,
                          notes=f"exponent systematic (half-spread over xs={list(spec['xs'])}): {syst_x:.4f}")

REC = bank(CUT, spec, ROWS, FSS, SWP, FREE, syst_x)
if WRITE_JSON:
    print("wrote", tf.save_record(REC, OUT / f"{CUT}.json"))
print(json.dumps({k: REC[k] for k in ("tag", "point", "fss", "quality")}, indent=1, default=tf._json_default))

## 7 · Batch: bank every registered cut

Headless pass over the whole registry (no figures) — run once after any data pull (with
`WRITE_JSON = True` to rebank the records). Prints the summary table.

In [ ]:
RUN_ALL = True

def run_cut(tag, verbose=False):
    sp = CUTS[tag]; d = load_cut(sp)
    fits = {L: tf.locate_all(c, sp["window"], ERR_MODE) for L, c in d["O"].items()}
    s2f = {L: (tf.fit_logistic(d["S2"][L], ERR_MODE, sp["window"]) if L in d["S2"] else None) for L in d["O"]}
    rows = []
    for L in fits:
        h_c, err, info = combine_markers(fits[L], s2f[L])
        rows.append({"L": L, "h_c": h_c, "h_c_err": err, **info,
                     "locators": {m: [f.h_c, f.h_c_err, f.chi2red] for m, f in fits[L].items()},
                     "S2": None if s2f[L] is None or not s2f[L].ok() else [s2f[L].h_c, s2f[L].h_c_err, s2f[L].chi2red],
                     "n_points": int(len(d["O"][L].h)), "amp": float(fits[L]["logistic"].popt[1]) if fits[L]["logistic"].ok() else None})
    Ls = [r["L"] for r in rows if np.isfinite(r["h_c"])]; hc = [r["h_c"] for r in rows if np.isfinite(r["h_c"])]; he = [r["h_c_err"] for r in rows if np.isfinite(r["h_c"])]
    fss = tf.fss_fit(Ls, hc, he, sp["x_main"], ERR_MODE); swp = tf.fss_sweep(Ls, hc, he, sp["xs"])
    free = tf.fss_free(Ls, hc, he) if len(Ls) >= 3 else None
    sx = 0.5 * (max(s["h_inf"] for s in swp) - min(s["h_inf"] for s in swp)) if len(Ls) >= 2 else np.nan
    rec = bank(tag, sp, rows, fss, swp, free, sx)
    if WRITE_JSON: tf.save_record(rec, OUT / f"{tag}.json")
    return rec

if RUN_ALL:
    print(f"{'cut':34s} {'L':10s} {'h_c(L)':30s} {'h_c(inf)':>20s} {'±x':>7s} q   (q: ✓/✗ fits converged+rise, F = FSS ok)")
    for tag in CUTS:
        r = run_cut(tag)
        hcs = " ".join(f"{p['h_c']:.3f}" for p in r["per_L"])
        f = r["fss"]; q = r["quality"]
        hinf = f"{f['h_inf']:.4f}({f['h_inf_err']:.4f})" if f.get("h_inf") is not None else "   L=4 only    "
        flag = ("✓" if q["all_converged"] and q["rise_ok"] else "✗") + ("F" if q["fss_ok"] else "-")
        print(f"{tag:34s} {str([p['L'] for p in r['per_L']]):10s} {hcs:30s} {hinf:>20s} {r['syst_exponent'] if r['syst_exponent'] is None else round(r['syst_exponent'], 4)!s:>7} {flag}")
    print("banked ->", OUT)

## 8 · Tests

Cheap checks that the machinery does what it claims: a synthetic-logistic recovery test,
loader dedupe (lowest-energy run wins), and physics consistency against the exact
$h_y=0$ anchors — the magnetic FSS must be compatible with $h_x^c=1$; the electric FSS at
$h_x=0.2$ must land in the neighbourhood of $h_z^c(h_x{=}0)=0.194$ (not equal: $h_x\neq0$,
and the residual finite-size form is not exact) — and the old lane's known high bias.

In [ ]:
fails = []
def check(name, ok, detail=""):
    d = str(detail)
    print(("PASS " if ok else "FAIL ") + name + (f"  [{d}]" if d else ""))
    if not ok: fails.append(name)

# T1 synthetic recovery: logistic + noise, inflection recovered within 2 sigma (PDG-inflated)
rng = np.random.default_rng(1); h = np.linspace(0.1, 0.5, 15); ye = np.full_like(h, 0.01)
y = tf.logistic(h, 0.0, 0.85, 0.283, 0.03) + rng.normal(0, 1, len(h)) * ye
f = tf.fit_logistic(tf.Curve(4, h, y, ye, "O_FM"))
check("T1 synthetic logistic recovers h0", abs(f.h_c - 0.283) < 2.5 * f.h_c_err, f"{f.h_c:.4f}±{f.h_c_err:.4f}")

# T2 synthetic FSS: h(L)=0.2+0.8 L^-1.587 recovered at fixed x and free x
LL = np.array([4, 5, 6, 7, 8]); hh = 0.2 + 0.8 * LL ** (-1 / NU); ee = np.full(len(LL), 0.003)
s = tf.fss_fit(LL, hh, ee, 1 / NU); fr = tf.fss_free(LL, hh + rng.normal(0, 1, 5) * ee, ee)
check("T2 synthetic FSS fixed-x exact", abs(s["h_inf"] - 0.2) < 1e-9 and abs(s["a"] - 0.8) < 1e-9)
check("T2 synthetic FSS free-x", abs(fr["x"] - 1 / NU) < 3 * fr["x_err"] + 0.05, f"x={fr['x']:.3f}±{fr['x_err']:.3f}")

# T3 loader dedupe on the prod electric cut: 15 points per L, and at each duplicated point the winner is the lower E
sp = CUTS["hy0_hx0.2_sweep-hz"]; O = tf.load_runs(sp["runs"], "hz", {"hx": 0.2, "hy": 0.0}, "O_FM_paratoric")
check("T3 15 points at L=4,5,6", all(len(O[L].h) == 15 for L in (4, 5, 6)), {L: len(O[L].h) for L in O})
allE = {}
for d in sp["runs"]:
    for fjs in Path(d).glob("*.json"):
        if fjs.name.endswith((".snapshots.json", ".curve.json")): continue
        j = json.loads(fjs.read_text()); c, o = j.get("config", {}), j.get("observables", {})
        if j.get("diverged") or o.get("O_FM_paratoric") is None: continue
        allE.setdefault((c["L"], round(c["hz"], 6)), []).append(o["E0"])
dupes = {k: v for k, v in allE.items() if len(v) > 1}
ok = all(abs(O[L].E[list(np.round(O[L].h, 6)).index(hz)] - min(v)) < 1e-9 for (L, hz), v in dupes.items())
check("T3 lowest-E wins at every duplicated point", ok, f"{len(dupes)} duplicated points")

# T4 physics: hy=0 magnetic FSS compatible with the exact first-order hx_c = 1 (hz=0.1 is nearby, not identical)
r = run_cut("hy0_hz0.1_sweep-hx"); f = r["fss"]
check("T4 magnetic h_c(inf) within 3 sigma of 1.0", abs(f["h_inf"] - 1.0) < 3 * f["h_inf_err"] + (r["syst_exponent"] or 0), f"{f['h_inf']:.3f}±{f['h_inf_err']:.3f}")
# T5 physics: hy=0 electric FSS at hx=0.2 lands near the hx=0 anchor 0.194 (window [0.15, 0.27])
r = run_cut("hy0_hx0.2_sweep-hz"); f = r["fss"]
check("T5 electric h_c(inf) in [0.15, 0.27]", 0.15 < f["h_inf"] < 0.27, f"{f['h_inf']:.3f}±{f['h_inf_err']:.3f} (x={f['x']:.3f})")
check("T5 L=4 pseudo-critical near the campaign half-max 0.29", abs(r["per_L"][0]["h_c"] - 0.29) < 0.03, f"{r['per_L'][0]['h_c']:.4f}")
# T6 monotone trend: h_c(L) decreases with L on the 2nd-order cut, increases on the 1st-order cut (OBC)
r2 = run_cut("hy0_hx0.2_sweep-hz"); r1 = run_cut("hy0_hz0.1_sweep-hx")
check("T6 electric h_c(L) decreasing", all(np.diff([p["h_c"] for p in r2["per_L"]]) < 0))
check("T6 magnetic h_c(L) increasing", all(np.diff([p["h_c"] for p in r1["per_L"]]) > 0))
# T7 hy shifts the electric L=4 point down (hy_cuts finding: 0.290/0.284/0.267 half-max)
pts = [run_cut(f"hy{hy:g}_hx0.2_sweep-hz")["per_L"][0]["h_c"] for hy in (0.0, 0.2, 0.4)]
check("T7 electric L=4 h_c decreases with hy", pts[0] > pts[1] > pts[2], np.round(pts, 4))
if DATA.exists():   # T8 old lane: hz=0 magnetic ladder L=4..7 compatible with the exact 1.0
    r = run_cut("hy0_hz0_sweep-hx@old"); f = r["fss"]
    check("T8 old-lane hz=0 magnetic h_c(inf) within 3 sigma of 1.0", abs(f["h_inf"] - 1.0) < 3 * f["h_inf_err"] + (r["syst_exponent"] or 0), f"{f['h_inf']:.3f}±{f['h_inf_err']:.3f}")
    r = run_cut("hy0_hx0_sweep-hz@old"); f = r["fss"]
    print(f"INFO old-lane hx=0 electric h_c(inf) = {f['h_inf']:.3f}±{f['h_inf_err']:.3f} vs exact 0.194 -> bias {f['h_inf'] - 0.193869:+.3f} (known: pre-optimization stack)")
print("\nALL PASS" if not fails else f"\n{len(fails)} FAILED: {fails}")
assert not fails